In [ ]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')


In [ ]:
import asyncio
import collections
import concurrent.futures
import json
import re
import time
import litellm
import pandas as pd
import torch
import yaml
from pathlib import Path
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import EvalConfig, build_submission_limits, ALLOWED_ADAPTER_EXTENSIONS
from swegemma.evaluate import Evaluator
from swegemma.models import load_tasks

litellm.drop_params = True

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
DECLARED_MODEL = 'gemma-4-e4b-it-qat-w4a16-ct'
MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{DECLARED_MODEL}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}'

SOUND12_IDS = ["fastapi_14297", "fastapi_14301", "fastapi_14360", "fastapi_14361", "fastapi_14372", "fastapi_14458", "fastapi_14463", "fastapi_14786", "httpx_3672", "requests_6644", "requests_7205", "requests_7309", "requests_7427", "requests_7505", "rich_2725", "rich_2943", "rich_3006", "rich_3130", "rich_3296", "rich_3506", "rich_3518", "rich_3718", "rich_3894", "rich_3905"]
VARIANTS = {
  "v2c_control": {
    "agent.yaml": "agent_class: SequentialAgent\nname: v2c_seq2_t0_t0_agent\ndescription: Two-stage fresh-context SWE repair agent (Stage 1 coder without submit_patch -> Stage 2 adversarial verifier/fixer without write_file).\nsub_agents:\n  - config_path: sub_agents/coder.yaml\n  - config_path: sub_agents/verifier.yaml\n",
    "configs/sampling_s1.yaml": "temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 4096\nthinking_config:\n  thinking_budget: 0\n  include_thoughts: false\n",
    "configs/sampling_s2.yaml": "temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 4096\nthinking_config:\n  thinking_budget: 0\n  include_thoughts: false\n",
    "eval_config.yaml": "evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 36\n  max_time_minutes: 8\n  max_turns: 80\n",
    "prompts/coder.md": "You are Stage 1 (Coder) in `/workspace`.\nYour ONLY available tools are: `run_command`, `read_file`, `edit_file`, `write_file`, `get_status`. Always invoke shell commands through `run_command(command=...)`.\nGoal: quickly locate the exact source file and apply a clean initial fix in ~10\u201314 tool calls, then hand off to Stage 2.\n\n## Steps\n1. **Locate (calls 1\u20135)**:\n   - Call `run_command` with a targeted search such as `git grep -n -F -e 'SymbolName' -- '*.py' | head -25`.\n   - Call `read_file` on the relevant 80\u2013120 line slice (never dump huge files).\n2. **Edit source code (by call 10)**:\n   - Call `edit_file` with a short, unique `old_string` and `allow_multiple=false`.\n   - For FastAPI doc-code issues, edit executable files under `docs_src/`.\n   - If `edit_file` fails twice, call `run_command` with a short inline Python exact-replace script and compile check.\n   - NEVER call `write_file` to create test or scratch scripts (never pass `/tmp/...` to `write_file`\u2014`write_file` writes inside `/workspace`).\n3. **Compile & hand off (calls 11\u201314)**:\n   - Call `run_command` with `python3 -m py_compile <edited_file>` and optionally one short inline `python3 -c \"...\"` check.\n   - Stop as soon as the edit compiles and output a 4-line handoff summary: edited files, root cause, change made, and edge cases to check.\n\n## Hard Rules\n- NEVER modify test files (`tests/`, `test_*.py`, `*_test.py`) or configs (`conftest.py`, `pytest.ini`, `pyproject.toml`, `setup.cfg`, `tox.ini`).\n- NEVER run bare `pytest` or `pip install`.\n",
    "prompts/verifier.md": "You are Stage 2 (Adversarial Verifier & Fixer) with a fresh context in `/workspace`.\nYour ONLY available tools are: `run_command`, `read_file`, `edit_file`, `get_status`, `submit_patch`. Always invoke shell commands through `run_command(command=...)`.\nStage 1 often edits the right file with a subtle bug, missed edge case, wrong exception/message, or incomplete logic. Audit critically, fix any flaw, clean stray files, and call `submit_patch`.\n\n## Problem Statement\n{problem_description}\n\n## Stage 1 Handoff\n{coder_notes?}\n\n## Steps\n1. **Audit current diff (call 1)**:\n   - Call `run_command` with `git status -s && git diff`.\n   - If the diff is empty, locate the source file immediately (`run_command` with `git grep`) and implement the fix with `edit_file`.\n   - If any untracked scratch files (`tmp/`, `verify_fix.py`, `repro.py`, `test_*.py`) or edits to protected test/config files exist in `/workspace`, clean/revert them via `run_command` (`rm -rf` / `git checkout --`).\n2. **Inspect full context (calls 2\u20134)**:\n   - Call `read_file` on 80\u2013120 lines around every edited function.\n   - Verify EVERY requirement in the problem statement: exact return value/type, exception type & message, default args, None/empty/boundary edge cases, and sibling callers.\n3. **Run focused verification (calls 5\u20138)**:\n   - Call `run_command` to run `python3 -m py_compile <edited_files>` AND a short inline `python3 -c \"...\"` script testing the reported bug + 2 edge cases + 1 normal control case.\n   - You may also run ONE targeted pytest node (`pytest tests/path/test_file.py -k <name>`) via `run_command`. If `pytest` is disabled, do NOT retry it\u2014use inline `python3 -c` assertions.\n   - NEVER write test files inside `/workspace` and NEVER loop more than twice debugging your own check script.\n4. **Repair if needed & submit**:\n   - If your audit or check reveals ANY defect, fix the source file with `edit_file` and re-run `py_compile` + inline check.\n   - Verify `git status -s` shows ONLY intended source edits (no stray files, no test/config changes).\n   - Call `submit_patch` as your final tool action while budget remains, confirm `patch_size > 0` and `files_changed > 0`, then give a 1-sentence summary.\n\n## Hard Rules\n- NEVER modify test files (`tests/`, `test_*.py`, `*_test.py`) or configs (`conftest.py`, `pytest.ini`, `pyproject.toml`, `setup.cfg`, `tox.ini`).\n- NEVER run bare `pytest`, `pip install`, or `git commit`.\n- NEVER finish without calling `submit_patch` on a compiled, non-empty source fix.\n",
    "sub_agents/coder.yaml": "name: coder_agent\nmodel: gemma-4-e4b-it-qat-w4a16-ct\ndescription: Stage 1 coder that locates the root cause and applies an initial source fix in /workspace without submitting.\ninstruction: !include ../prompts/coder.md\noutput_key: coder_notes\ndisallow_transfer_to_parent: true\ndisallow_transfer_to_peers: true\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\ngenerate_content_config: !include ../configs/sampling_s1.yaml\n",
    "sub_agents/verifier.yaml": "name: verifier_fixer_agent\nmodel: gemma-4-e4b-it-qat-w4a16-ct\ndescription: Stage 2 fresh-context adversarial verifier and fixer that audits git diff, runs focused inline checks, repairs bugs, cleans scratch files, and calls submit_patch.\ninstruction: !include ../prompts/verifier.md\ninclude_contents: none\ndisallow_transfer_to_parent: true\ndisallow_transfer_to_peers: true\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - get_status\n  - submit_patch\ngenerate_content_config: !include ../configs/sampling_s2.yaml\n"
  },
  "v6_seq2_clean": {
    "agent.yaml": "agent_class: SequentialAgent\nname: gemma4_e4b_v6_seq2_clean\ndescription: Two-stage fresh-context E4B agent (Coder without submit_patch -> Verifier with real targeted pytest & heredoc edit recovery).\nsub_agents:\n  - config_path: sub_agents/coder.yaml\n  - config_path: sub_agents/verifier.yaml\n",
    "configs/sampling.yaml": "temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 4096\nthinking_config:\n  thinking_budget: 0\n  include_thoughts: false\n",
    "eval_config.yaml": "evaluation:\n  timeout_seconds: 240\n  max_tool_calls: 60\n  max_time_minutes: 8\n  max_turns: 120\n",
    "prompts/coder.md": "You are Stage 1 (Coder) in `/workspace`.\nYour ONLY tools are: `run_command`, `read_file`, `edit_file`, `write_file`, `get_status`. Always run shell commands via `run_command(command=...)`.\nGoal: locate the root cause, apply a minimal clean fix to SOURCE code in `/workspace`, test it against the real repo test file, and hand off to Stage 2 (target ~15\u201322 calls).\n\n## Steps\n1. **Locate source + relevant existing test file (calls 1\u20135)**:\n   - Call `run_command` with `git grep -n -F -e '<Symbol1>' -e '<Symbol2>' -- '*.py' | head -25`.\n   - Find the matching existing test file via `run_command`: `find tests -name '*<module>*' | head -15`.\n   - Call `read_file` on the relevant 80\u2013120 line slice of the source file (and the relevant test if helpful). For FastAPI doc-code issues, edit executable files under `docs_src/`.\n2. **Edit source cleanly (reliable editing rules!)**:\n   - Copy `old_string` **verbatim** from your latest `read_file` or `sed -n 'A,Bp' <file>` output. Keep it short (3\u201312 lines), exact indentation, NO manually added `\\n` or `\\\"` escapes, `allow_multiple=false`.\n   - **After ONE failed `edit_file`**, do NOT guess `old_string` again! Immediately inspect exact lines with `sed -n 'A,Bp' <file>` or apply the edit via a single-quoted Python heredoc in `run_command`:\n     ```bash\n     python3 - <<'EOF'\n     from pathlib import Path\n     p = Path(\"path/to/file.py\")\n     s = p.read_text()\n     old = \"\"\"exact old block\"\"\"\n     new = \"\"\"exact new block\"\"\"\n     assert s.count(old) == 1, f\"count={s.count(old)}\"\n     p.write_text(s.replace(old, new, 1))\n     EOF\n     ```\n   - NEVER use multiline `python3 -c \"...\"` (quotes break). ALWAYS use `python3 - <<'EOF' ... EOF`.\n   - NEVER call `write_file` on `/tmp/...` or to create scratch/test scripts.\n3. **Compile & run real targeted repo tests**:\n   - Call `run_command` with `python3 -m py_compile <edited_file>` AND run the relevant existing test file: `pytest tests/path/to/test_file.py -x -q --tb=short` (or `-k <keyword>` if the file is huge).\n   - NEVER write fake mock re-implementations of the function under test\u2014always import and exercise the real `/workspace` module!\n   - If `pytest` is disabled in the sandbox, do NOT retry it\u2014run a real import check via `python3 - <<'EOF' ... EOF`.\n4. **Hand off concisely**:\n   - Once your edit compiles and passes (or after ~22 calls), stop and write a 5-line handoff: edited files, exact root cause, test command & output, and edge cases Stage 2 should check.\n\n## Hard Rules\n- NEVER modify test files (`tests/`, `test_*.py`, `*_test.py`) or configs (`conftest.py`, `pytest.ini`, `pyproject.toml`, `setup.cfg`, `tox.ini`).\n- NEVER run bare `pytest` or full-repo sweeps (`pytest .`). Always pass a specific test file path.\n- NEVER run `pip install` or leave untracked scratch files in `/workspace`.\n",
    "prompts/verifier.md": "You are Stage 2 (Adversarial Verifier & Fixer, fresh context) in `/workspace`.\nYour ONLY tools are: `run_command`, `read_file`, `edit_file`, `get_status`, `submit_patch`. Always run shell commands via `run_command(command=...)`.\nStage 1 often touches the right file with a subtle bug, missed edge case, wrong exception/message, or broken edit. Audit critically, test against the REAL codebase, fix any defect, and call `submit_patch`.\n\n## Problem Statement\n{problem_description}\n\n## Stage 1 Handoff\n{coder_notes?}\n\n## Steps\n1. **Audit current diff & clean workspace (call 1)**:\n   - Call `run_command` with `git status -s && git diff`.\n   - Clean any untracked scratch files (`rm -rf` on stray files) and revert any accidental edits to protected test/config files (`git checkout -- <file>`).\n   - If the diff is empty, locate the source file immediately (`git grep`) and implement the fix.\n2. **Read full surrounding context (calls 2\u20134)**:\n   - Call `read_file` on 80\u2013120 lines around every modified function.\n   - Check EVERY contract detail in the problem statement: exact return value/type, exception class & exact message, default args, None/empty/subclass/boundary inputs, and callers.\n3. **Run REAL targeted tests (not mocks!)**:\n   - First run `python3 -m py_compile <edited_files>` via `run_command`.\n   - Run the relevant existing repo test file: `pytest tests/path/to/test_file.py -x -q --tb=short` (find it via `find tests -name '*<module>*' | head -15` if Stage 1 did not name it).\n   - ALSO run one real-import edge-case check using a single-quoted heredoc `python3 - <<'EOF' ... EOF` that imports the actual modified function/class from `/workspace` and asserts the reported bug + 2 edge cases. (NEVER use multiline `python3 -c \"...\"`, and NEVER re-implement/mock the target function!)\n   - If `pytest` is disabled in the sandbox, rely on the real-import heredoc check\u2014do NOT retry `pytest`. Ignore unrelated pre-existing missing-fixture failures.\n4. **Reliable repair & submit**:\n   - If ANY test or edge case fails, repair the source file with `edit_file` (verbatim `old_string` from fresh `read_file`/`sed -n`, no manual escapes). After **ONE** failed `edit_file`, immediately use a single-quoted `python3 - <<'EOF' ... EOF` exact-replace script that asserts `s.count(old) == 1`.\n   - Re-run `py_compile` + the targeted `pytest` / heredoc check after your LAST edit.\n   - Verify `git status -s` shows ONLY intended source edits, then call `submit_patch` as your final tool action. Confirm `patch_size > 0` and `files_changed > 0`, and write a 1-sentence summary.\n\n## Hard Rules\n- NEVER modify test files (`tests/`, `test_*.py`, `*_test.py`) or configs (`conftest.py`, `pytest.ini`, `pyproject.toml`, `setup.cfg`, `tox.ini`).\n- NEVER run bare `pytest` (`pytest .`), `pip install`, or `git commit`.\n- NEVER finish without calling `submit_patch` on a compiled, non-empty source fix.\n",
    "sub_agents/coder.yaml": "name: coder_agent\nmodel: gemma-4-e4b-it-qat-w4a16-ct\ndescription: Stage 1 coder that locates the root cause, edits source code cleanly, verifies with real targeted repo tests, and hands off without submitting.\ninstruction: !include ../prompts/coder.md\noutput_key: coder_notes\ndisallow_transfer_to_parent: true\ndisallow_transfer_to_peers: true\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\ngenerate_content_config: !include ../configs/sampling.yaml\n",
    "sub_agents/verifier.yaml": "name: verifier_fixer_agent\nmodel: gemma-4-e4b-it-qat-w4a16-ct\ndescription: Stage 2 fresh-context adversarial verifier and fixer that audits git diff, runs real targeted repo pytest, fixes remaining bugs, and calls submit_patch.\ninstruction: !include ../prompts/verifier.md\ninclude_contents: none\ndisallow_transfer_to_parent: true\ndisallow_transfer_to_peers: true\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - get_status\n  - submit_patch\ngenerate_content_config: !include ../configs/sampling.yaml\n"
  }
}
tasks = load_tasks(TASKS_PATH)
wanted = set(SOUND12_IDS)
subset = sorted([t for t in tasks if t.instance_id in wanted], key=lambda t: t.instance_id)
assert len(subset) == len(SOUND12_IDS), f'Expected {len(SOUND12_IDS)}, got {len(subset)}'

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=False,
    max_loras=1,
    max_lora_rank=128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=(),
)
server_instance = VllmServer(vllm_cfg)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})', flush=True)

models = server_instance.create_model_registry(
    aliases=[DECLARED_MODEL],
    model_prefix='openai/',
    api_key=os.environ.get('LOCAL_API_KEY', 'EMPTY'),
)

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if (callable(coro_or_fn) and not asyncio.iscoroutine(coro_or_fn)) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

limits, gen_constraints = build_submission_limits()
all_summaries = []

for run_name, files_map in VARIANTS.items():
    print(f'\n================ RUNNING {run_name} ({len(subset)} tasks) ================', flush=True)
    run_dir = WORKING_DIR / run_name
    agent_dir = run_dir / 'agent'
    if agent_dir.exists():
        shutil.rmtree(agent_dir)
    for rel, content in files_map.items():
        p = agent_dir / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content, encoding='utf-8')

    ev_raw = yaml.safe_load((agent_dir / 'eval_config.yaml').read_text(encoding='utf-8')) or {}
    ev = ev_raw.get('evaluation', ev_raw)
    max_time_minutes = float(ev.get('max_time_minutes', 8.0))
    adapters = discover_adapters(str(agent_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

    eval_config = EvalConfig(
        tasks_path=TASKS_PATH,
        snapshots_dir=DATA_DIR / 'snapshots',
        results_dir=run_dir / 'results',
        submission_dir=agent_dir,
        models=models,
        sandbox='subprocess',
        timeout_seconds=int(ev.get('timeout_seconds', 240)),
        verification_timeout_seconds=int(ev.get('verification_timeout_seconds', 300)),
        max_time_minutes=max_time_minutes,
        max_tool_calls=int(ev.get('max_tool_calls', 28)),
        max_turns=int(ev.get('max_turns', 80)),
        limits=limits,
        generation_constraints=gen_constraints,
        adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(
            compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
        graph_dir=GRAPH_DIR,
        embeddings_dir=EMBEDDINGS_DIR,
        wheels_dir=DATA_DIR / 'wheels',
        verbose=False,
    )
    evaluator = Evaluator(eval_config)
    rows_path = run_dir / 'per_task.jsonl'
    rows = []
    for idx, task in enumerate(subset, start=1):
        started = time.perf_counter()
        try:
            result = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(subset))
            patch = (result.agent_patch or '').strip()
            row = dict(
                run=run_name, id=task.instance_id, repo=task.repo,
                resolved=bool(result.resolved), patch_chars=len(patch),
                tool_calls=result.tool_calls,
                agent_seconds=round(float(result.duration_seconds or 0.0), 1),
                test_exit_code=result.test_exit_code,
                error=getattr(result, 'error', None),
            )
        except Exception as exc:
            row = dict(
                run=run_name, id=task.instance_id, repo=task.repo,
                resolved=False, patch_chars=0, tool_calls=None,
                agent_seconds=None, test_exit_code=None, error=repr(exc)[:300],
            )
        row['wall_seconds'] = round(time.perf_counter() - started, 1)
        rows.append(row)
        with rows_path.open('a') as f:
            f.write(json.dumps(row) + '\n')
        print(f"[{run_name} {idx}/{len(subset)}] {row['id']}: resolved={row['resolved']} "
              f"patch={row['patch_chars']} calls={row['tool_calls']} "
              f"agent={row['agent_seconds']}s wall={row['wall_seconds']}s err={row['error']}", flush=True)

    df = pd.DataFrame(rows)
    agent_s = df['agent_seconds'].dropna()
    summary = {
        'run': run_name,
        'tasks': len(df),
        'resolved': int(df['resolved'].sum()),
        'resolution_rate': round(float(df['resolved'].mean()), 3),
        'no_patch': int((df['patch_chars'] == 0).sum()),
        'errors': int(df['error'].notna().sum()),
        'mean_tool_calls': round(float(df['tool_calls'].dropna().mean()), 1) if len(df['tool_calls'].dropna()) else None,
        'agent_seconds_median': float(agent_s.median()) if len(agent_s) else None,
        'wall_seconds_mean': round(float(df['wall_seconds'].mean()), 1),
    }
    (run_dir / 'summary.json').write_text(json.dumps(summary, indent=2))
    all_summaries.append(summary)
    print(json.dumps(summary, indent=2), flush=True)
    print(df.to_string(index=False), flush=True)

(WORKING_DIR / 'ablation_summary.json').write_text(json.dumps(all_summaries, indent=2))
print('\n=== FINAL ABLATION COMPARISON ===', flush=True)
print(pd.DataFrame(all_summaries).to_string(index=False), flush=True)
